# Capítulo 4 — Identificación de regímenes de mercado

Este notebook prepara la base para identificar regímenes de mercado del IBEX 35 mediante un modelo de Markov Switching.

**Nota:** en esta versión solo se construye la estructura inicial y el análisis exploratorio mínimo.

## 1. Configuración y rutas

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.6f}".format)

plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

In [ ]:
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures" / "chapter4"
TABLES_DIR = PROJECT_ROOT / "tables" / "chapter4"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("TABLES_DIR:", TABLES_DIR)

## 2. Carga de datos

In [ ]:
INPUT_FILE = PROCESSED_DATA_DIR / "dataset_cap4_input.csv"

df = pd.read_csv(INPUT_FILE, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Shape:", df.shape)
print("Rango de fechas:", df["date"].min(), "->", df["date"].max())
print("Columnas:", list(df.columns))

## 3. Selección de variables

In [ ]:
model_columns = ["ret_1d", "ewma_vol", "cvar_95_60d"]

df_model = df[["date", "sample_split_main"] + model_columns].copy()
df_model = df_model.dropna().reset_index(drop=True)

print("Shape df_model:", df_model.shape)
print("Missing por columna:")
print(df_model.isna().sum())

## 4. Split train/test

In [ ]:
if "sample_split_main" not in df_model.columns:
    raise ValueError("La columna 'sample_split_main' no está disponible en el dataset de entrada.")

df_train = df_model[df_model["sample_split_main"] == "train"].copy()
df_test = df_model[df_model["sample_split_main"] == "test"].copy()

print("Tamaño train:", df_train.shape)
print("Tamaño test:", df_test.shape)
print("Fechas train:", df_train["date"].min(), "->", df_train["date"].max())
print("Fechas test:", df_test["date"].min(), "->", df_test["date"].max())

## 5. Análisis exploratorio mínimo

In [ ]:
print("Descriptivos (train):")
display(df_train[model_columns].describe().T)

print("Descriptivos (test):")
display(df_test[model_columns].describe().T)

In [ ]:
corr_matrix = df_model[model_columns].corr()

print("Correlación entre variables:")
display(corr_matrix)

plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm", center=0)
plt.title("Matriz de correlación (variables del modelo)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 8), sharex=True)

for i, col in enumerate(model_columns):
    axes[i].plot(df_model["date"], df_model[col], linewidth=0.8)
    axes[i].set_title(col)
    axes[i].set_ylabel(col)

axes[-1].set_xlabel("Fecha")
plt.tight_layout()
plt.show()

## 6. Placeholder de modelo

In [ ]:
# 1) Serie objetivo (solo train y solo ret_1d)
endog_train = df_train["ret_1d"].astype(float)

# 2) Modelo 1: Markov Switching con 2 regímenes
msm_2 = sm.tsa.MarkovRegression(
    endog_train,
    k_regimes=2,
    trend="c",
    switching_variance=True,
)
res_2 = msm_2.fit(disp=False)

print("=" * 80)
print("MODELO 1 - Markov Switching (2 regímenes)")
print("=" * 80)
print(res_2.summary())
print(f"Log-likelihood: {res_2.llf:,.6f}")
print(f"AIC: {res_2.aic:,.6f}")
print(f"BIC: {res_2.bic:,.6f}")

# 3) Modelo 2: Markov Switching con 3 regímenes
msm_3 = sm.tsa.MarkovRegression(
    endog_train,
    k_regimes=3,
    trend="c",
    switching_variance=True,
)
res_3 = msm_3.fit(disp=False)

print("\n" + "=" * 80)
print("MODELO 2 - Markov Switching (3 regímenes)")
print("=" * 80)
print(res_3.summary())
print(f"Log-likelihood: {res_3.llf:,.6f}")
print(f"AIC: {res_3.aic:,.6f}")
print(f"BIC: {res_3.bic:,.6f}")

# 4) Comparación simple entre modelos
comparison_df = pd.DataFrame(
    {
        "model": ["MS(2)", "MS(3)"],
        "k_regimes": [2, 3],
        "log_likelihood": [res_2.llf, res_3.llf],
        "aic": [res_2.aic, res_3.aic],
        "bic": [res_2.bic, res_3.bic],
    }
)

print("\nComparación de modelos:")
display(comparison_df)

# 5) Selección por mejor AIC/BIC (menor valor)
best_aic_idx = comparison_df["aic"].idxmin()
best_bic_idx = comparison_df["bic"].idxmin()

if best_aic_idx == best_bic_idx:
    selected_idx = best_aic_idx
else:
    selected_idx = best_bic_idx

selected_model_name = comparison_df.loc[selected_idx, "model"]
selected_res = res_2 if selected_model_name == "MS(2)" else res_3

print(
    f"Modelo seleccionado: {selected_model_name} "
    f"(mejor AIC: {comparison_df.loc[best_aic_idx, 'model']}, "
    f"mejor BIC: {comparison_df.loc[best_bic_idx, 'model']})"
)

# 6) Probabilidades suavizadas y régimen más probable
smoothed_probs = selected_res.smoothed_marginal_probabilities.copy()
smoothed_probs.columns = [f"p_regime_{i}" for i in smoothed_probs.columns]

most_likely_regime = smoothed_probs.values.argmax(axis=1)

# Agregar al dataframe de train (sin usar test)
df_train = df_train.copy()
df_train["regime"] = most_likely_regime

print("\nVista rápida de df_train con régimen asignado:")
display(df_train[["date", "ret_1d", "regime"]].head())

print("\nDistribución de regímenes en train:")
print(df_train["regime"].value_counts(dropna=False).sort_index())


## 7. Interpretación económica de los regímenes (solo train)

Análisis descriptivo por régimen sin modificar el modelo ni usar `df_test`.

In [ ]:
analysis_cols = ["ret_1d", "ewma_vol", "cvar_95_60d", "regime"]
missing_cols = [c for c in analysis_cols if c not in df_train.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas requeridas en df_train: {missing_cols}")

summary_by_regime = (
    df_train.groupby("regime", dropna=False)
    .agg(
        n_obs=("regime", "size"),
        mean_ret_1d=("ret_1d", "mean"),
        mean_ewma_vol=("ewma_vol", "mean"),
        mean_cvar_95_60d=("cvar_95_60d", "mean"),
    )
    .reset_index()
)

summary_by_regime["frequency_pct"] = (
    summary_by_regime["n_obs"] / summary_by_regime["n_obs"].sum() * 100
)

summary_by_regime = summary_by_regime[
    [
        "regime",
        "n_obs",
        "frequency_pct",
        "mean_ret_1d",
        "mean_ewma_vol",
        "mean_cvar_95_60d",
    ]
]

summary_risk_order = summary_by_regime.sort_values(
    by=["mean_ewma_vol", "mean_cvar_95_60d"],
    ascending=[True, True],
).reset_index(drop=True)

print("Tabla resumen por régimen:")
display(summary_by_regime)

print("\nTabla ordenada por nivel de riesgo (menor a mayor):")
display(summary_risk_order)

fig, axes = plt.subplots(ncols=2, figsize=(14, 5))

sns.boxplot(data=df_train, x="regime", y="ewma_vol", ax=axes[0])
axes[0].set_title("EWMA volatility por régimen")
axes[0].set_xlabel("Régimen")
axes[0].set_ylabel("ewma_vol")

sns.boxplot(data=df_train, x="regime", y="cvar_95_60d", ax=axes[1])
axes[1].set_title("CVaR 95% (60d) por régimen")
axes[1].set_xlabel("Régimen")
axes[1].set_ylabel("cvar_95_60d")

plt.tight_layout()
plt.show()

plot_df = df_train.sort_values("date").copy()

fig, ax = plt.subplots(figsize=(14, 5))
sns.scatterplot(
    data=plot_df,
    x="date",
    y="ret_1d",
    hue="regime",
    palette="tab10",
    alpha=0.85,
    s=18,
    ax=ax,
)
ax.set_title("ret_1d en el tiempo, coloreado por régimen")
ax.set_xlabel("Fecha")
ax.set_ylabel("ret_1d")
ax.legend(title="Régimen", bbox_to_anchor=(1.01, 1), loc="upper left")

plt.tight_layout()
plt.show()